# T09: YAMNet-upotukset (malli C1)

Tämä notebook ajaa vain koodia, jonka logiikka on `src/yamnet.py`:ssä. Sisältö:

1. Data ja YAMNet (`hub.load`, ei `hub.KerasLayer`, koska Keras 3)
2. Kehyskohtaiset upotukset kaikille 400 leikkeelle `.npy`-välimuistiin (Drive)
3. Järkevyystesti: YAMNetin oma AudioSet-tunniste luokittain

C1 käyttää YAMNetin omaa esikäsittelyä (16 kHz, mono), ei T04:n log-meliä (PLAN.md). YAMNet on jäädytetty, joten upotukset lasketaan kaikille leikkeille ilman datavuotoa. Luokitin ja mahdollinen normalisointi opetetaan T10:ssä vain opetusfoldeista.

In [ ]:
import sys
from pathlib import Path

# Repon juuri polkuun, jotta `src` löytyy (notebook ajetaan kansiosta notebooks/).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import collections
import csv
import numpy as np
import pandas as pd
import keras

from src.data import ANIMAL_CLASSES, download_esc50, load_metadata
from src.yamnet import build_yamnet_cache, load_wave_16k, load_yamnet

# Colabissa: DATA_DIR = "/content/drive/MyDrive/Elainaanet-CNN/data"
DATA_DIR = ROOT / "data"
keras.utils.set_random_seed(42)   # YAMNet on deterministinen; siemen projektin käytännön mukaan

## 1. Data ja YAMNet

In [ ]:
download_esc50(DATA_DIR)          # ei lataa uudelleen, jos data on jo paikallaan
df = load_metadata(DATA_DIR)
model = load_yamnet()

wave = load_wave_16k(df["path"][0])
print(wave.shape, wave.dtype, f"min {wave.min():.3f}, max {wave.max():.3f}")

## 2. Upotukset välimuistiin

Muoto (leikkeet, kehykset, 1024). YAMNetin kehys on 0,96 s ja hyppy 0,48 s. Toinen kutsu lukee välimuistin.

In [ ]:
x = build_yamnet_cache(df, Path(DATA_DIR) / "cache", model=model)
print(x.shape, x.dtype, f"{x.nbytes / 1e6:.1f} Mt", "NaN:", np.isnan(x).any())
print("välimuisti luettu uudelleen, sama:", np.array_equal(x, build_yamnet_cache(df, Path(DATA_DIR) / "cache")))

## 3. Järkevyystesti

YAMNetin oma ennuste (AudioSetin 521 luokkaa) leikkeen kehysten keskiarvopisteistä, kolme yleisintä tunnistetta luokittain. Jos tunnisteet vastaavat eläimiä, esikäsittely (16 kHz, mono, −1…1) on oikein. **Tämä ei ole C1:n tulos**, koska luokitin opetetaan vasta T10:ssä.

In [ ]:
with open(model.class_map_path().numpy().decode(), encoding="utf-8") as f:
    names = [row["display_name"] for row in csv.DictReader(f)]

top = []
for path in df["path"]:
    scores, _, _ = model(load_wave_16k(path))
    top.append(names[int(scores.numpy().mean(axis=0).argmax())])
df["yamnet_top"] = top

rows = {cls: ", ".join(f"{n} {k}" for n, k in collections.Counter(
            df.loc[df["target"] == t, "yamnet_top"]).most_common(3))
        for t, cls in enumerate(ANIMAL_CLASSES)}
pd.Series(rows, name="YAMNetin yleisimmät tunnisteet (40 leikettä)").to_frame()